<a href="https://colab.research.google.com/github/bipana-aa/Numerical-Methods/blob/main/Chapter_2_Exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Numerical Methods: Chapter 2 Exercises
**Name:** Bipana Khadka
 **Course:** BSIT400

## CA2.2 Inverse of lower and upper triangular matrices (Gauss-Jordan)

To find the inverse of a matrix T with Gauss-Jordan, I write T next to the identity matrix, $[T \mid I]$, and do row operations until the left side becomes $I$. The right side is then $T^{-1}$.

Since L and U are triangular, a lot of the work is already done. In a lower triangular matrix there is nothing above the diagonal, so I only have to clear the entries below each pivot. In an upper triangular matrix it is the other way around, so I start from the last row and clear the entries above each pivot.

**Pseudocode (lower triangular L):**
1. Start with `inv` as the identity matrix.
2. For k = 1 to n:
   1. If L[k][k] is 0, stop, because the matrix has no inverse.
   2. Divide row k of `inv` by L[k][k].
   3. For every row i below k, subtract L[i][k] times row k from row i of `inv`.
3. Return `inv`.

**Pseudocode (upper triangular U):** the same steps, but go from k = n down to 1 and fix the rows above k.

Because we skip the zero parts, the cost is about n^3/6 multiplications, much less than inverting a full matrix.

In [4]:
def inverse_lower(L, tol=1e-12):
    n = len(L)
    inv = [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]
    for k in range(n):
        piv = L[k][k]
        if abs(piv) < tol:
            raise ValueError(f"Zero on the diagonal at ({k+1},{k+1}): matrix is singular")
        for j in range(k + 1):              # make the pivot 1 by dividing row k
            inv[k][j] /= piv
        for i in range(k + 1, n):           # clear column k below the pivot
            f = L[i][k]
            if f != 0.0:
                for j in range(k + 1):
                    inv[i][j] -= f * inv[k][j]
    return inv


def inverse_upper(U, tol=1e-12):
    n = len(U)
    inv = [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]
    for k in range(n - 1, -1, -1):
        piv = U[k][k]
        if abs(piv) < tol:
            raise ValueError(f"Zero on the diagonal at ({k+1},{k+1}): matrix is singular")
        for j in range(k, n):               # make the pivot 1 by dividing row k
            inv[k][j] /= piv
        for i in range(k - 1, -1, -1):      # clear column k above the pivot
            f = U[i][k]
            if f != 0.0:
                for j in range(k, n):
                    inv[i][j] -= f * inv[k][j]
    return inv

### Testing with a small example
To check the answer, I multiply each matrix by its inverse. If the inverse is right, the result should be the identity matrix, so the error should be 0 (or very close to it).

In [5]:
import random

def matmul(A, B):
    n = len(A)
    return [[sum(A[i][k] * B[k][j] for k in range(n)) for j in range(n)] for i in range(n)]

def max_err_from_identity(P):
    n = len(P)
    return max(abs(P[i][j] - (1.0 if i == j else 0.0)) for i in range(n) for j in range(n))

def show(name, M):
    print(name)
    for row in M:
        print("  " + "  ".join(f"{x:9.5f}" for x in row))

L = [[2.0, 0, 0], [1, 4, 0], [3, 5, 8]]
U = [[2.0, 1, 3], [0, 4, 5], [0, 0, 8]]

Li, Ui = inverse_lower(L), inverse_upper(U)
show("L =", L); show("inv(L) =", Li)
print("max |L*inv(L) - I| =", max_err_from_identity(matmul(L, Li)), "\n")
show("U =", U); show("inv(U) =", Ui)
print("max |U*inv(U) - I| =", max_err_from_identity(matmul(U, Ui)))

L =
    2.00000    0.00000    0.00000
    1.00000    4.00000    0.00000
    3.00000    5.00000    8.00000
inv(L) =
    0.50000    0.00000    0.00000
   -0.12500    0.25000    0.00000
   -0.10938   -0.15625    0.12500
max |L*inv(L) - I| = 0.0 

U =
    2.00000    1.00000    3.00000
    0.00000    4.00000    5.00000
    0.00000    0.00000    8.00000
inv(U) =
    0.50000   -0.12500   -0.10938
    0.00000    0.25000   -0.15625
    0.00000    0.00000    0.12500
max |U*inv(U) - I| = 0.0


### Random matrices and a singular case
I also tried random matrices of different sizes, and one matrix with a zero on the diagonal to see that the program catches it.

In [6]:
def random_triangular(n, lower=True):
    M = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if (lower and j <= i) or (not lower and j >= i):
                M[i][j] = random.uniform(-3, 3)
        M[i][i] = random.choice([-1, 1]) * random.uniform(1, 4)   # keep the diagonal away from 0
    return M

random.seed(1)
for n in (1, 2, 5, 10, 25):
    L, U = random_triangular(n, True), random_triangular(n, False)
    eL = max_err_from_identity(matmul(L, inverse_lower(L)))
    eU = max_err_from_identity(matmul(U, inverse_upper(U)))
    print(f"n={n:2d}  lower err={eL:.2e}  upper err={eU:.2e}")

try:
    inverse_lower([[1.0, 0.0], [2.0, 0.0]])
except ValueError as e:
    print("\nSingular test:", e)

n= 1  lower err=0.00e+00  upper err=0.00e+00
n= 2  lower err=0.00e+00  upper err=1.11e-16
n= 5  lower err=1.11e-16  upper err=1.67e-16
n=10  lower err=9.99e-16  upper err=3.64e-16
n=25  lower err=1.93e-13  upper err=4.86e-15

Singular test: Zero on the diagonal at (2,2): matrix is singular


### Conclusion
The inverses of both the lower and upper triangular matrices came out correct. The errors are around 1e-13 or smaller, which is just normal rounding error in floating point numbers. The program also gives a clear message when a diagonal entry is zero, since such a matrix has no inverse.